In [1]:
# importing relevant modules 
import warnings
warnings.filterwarnings("ignore")

In [2]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

In [3]:
# imporing relevant modules required to ingest data into kafka_broker
from pyspark.storagelevel import StorageLevel
from pyspark.sql.types import *
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, TimestampType


In [4]:
spark = SparkSession.builder \
    .appName("Kafka to File") \
    .master("spark://localhost:7077") \
    .config("spark.executor.memory", "1g") \
    .config("spark.driver.memory", "1g") \
    .config("spark.submit.deployMode", "client") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.1") \
    .getOrCreate()

print("Connected to Spark cluster!")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/10 17:26:25 WARN Utils: Your hostname, Dhananjays-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 10.180.236.156 instead (on interface en0)
26/02/10 17:26:25 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/Users/dhananjayojha/Documents/projects/de_projects/nyc_taxi_cc/.venv/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/dhananjayojha/.ivy2.5.2/cache
The jars for the packages stored in: /Users/dhananjayojha/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-dbddaf8e-53b3-4f26-9564-f4c91ca432d7;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.1.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4

Connected to Spark cluster!


In [5]:
print("Connected to Spark cluster!")
print(f"Master: {spark.sparkContext.master}")

# Define schema matching your green taxi data
schema = StructType([
    StructField("VendorID", LongType(), True),
    StructField("lpep_pickup_datetime", StringType(), True),
    StructField("lpep_dropoff_datetime", StringType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("RatecodeID", DoubleType(), True),
    StructField("PULocationID", LongType(), True),
    StructField("DOLocationID", LongType(), True),
    StructField("passenger_count", DoubleType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("ehail_fee", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("payment_type", DoubleType(), True),
    StructField("trip_type", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True)
])

Connected to Spark cluster!
Master: spark://localhost:7077


In [6]:
print("\nReading from Kafka...")

df = spark.read \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "redpanda-1:29092") \
    .option("subscribe", "yellow_taxi_bookings") \
    .option("startingOffsets", "earliest") \
    .load() \
    .limit(5000)

print("Parsing...")
parsed_df = df.select(
    from_json(col("value").cast("string"), schema).alias("data")
).select("data.*")
    
print("Saving...")
parsed_df.write.mode("overwrite").parquet("/tmp/data/green_taxi/kafka_output")



Reading from Kafka...
Parsing...
Saving...


26/02/10 17:27:47 WARN ClientUtils: Couldn't resolve server redpanda-1:29092 from bootstrap.servers as DNS resolution failed for redpanda-1


Py4JJavaError: An error occurred while calling o64.parquet.
: org.apache.kafka.common.KafkaException: Failed to create new KafkaAdminClient
	at org.apache.kafka.clients.admin.KafkaAdminClient.createInternal(KafkaAdminClient.java:561)
	at org.apache.kafka.clients.admin.Admin.create(Admin.java:147)
	at org.apache.spark.sql.kafka010.ConsumerStrategy.createAdmin(ConsumerStrategy.scala:49)
	at org.apache.spark.sql.kafka010.ConsumerStrategy.createAdmin$(ConsumerStrategy.scala:46)
	at org.apache.spark.sql.kafka010.SubscribeStrategy.createAdmin(ConsumerStrategy.scala:102)
	at org.apache.spark.sql.kafka010.KafkaOffsetReaderAdmin.admin(KafkaOffsetReaderAdmin.scala:69)
	at org.apache.spark.sql.kafka010.KafkaOffsetReaderAdmin.fetchPartitionOffsets(KafkaOffsetReaderAdmin.scala:130)
	at org.apache.spark.sql.kafka010.KafkaOffsetReaderAdmin.getOffsetRangesFromUnresolvedOffsets(KafkaOffsetReaderAdmin.scala:379)
	at org.apache.spark.sql.kafka010.KafkaRelation.buildScan(KafkaRelation.scala:69)
	at org.ap

26/02/10 17:51:47 ERROR TaskSchedulerImpl: Lost executor 0 on 172.19.0.5: Worker shutting down
26/02/10 17:51:48 WARN StandaloneAppClient$ClientEndpoint: Connection to 172.19.0.4:7077 failed; waiting for master to reconnect...
26/02/10 17:51:48 WARN StandaloneSchedulerBackend: Disconnected from Spark cluster! Waiting for reconnection...


In [ ]:
print("✓ Done!")
spark.stop()